In [ ]:
!pip install ultralytics==8.4.7 markdown rich wrapt pandas mlflow huggingface_hub opencv-python wandb datasets -q

In [1]:
from datasets import load_dataset
import ultralytics
import os
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset, DatasetDict, ClassLabel, Audio as DatasetAudio
from PIL import Image
import shutil
from tqdm.auto import tqdm
from ultralytics import YOLO
import yaml
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import matplotlib.pyplot as plt
import librosa.display



/home/pierre/Documents/Projects/PST4/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ultralytics.checks()

Ultralytics 8.4.7 🚀 Python-3.13.9 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 7783MiB)
Setup complete ✅ (24 CPUs, 30.5 GB RAM, 806.2/929.9 GB disk)


In [3]:
SAMPLING_RATE = 16000
LABELS = ['other', 'drone']

# Paths
SPECTROGRAM_DIR = Path("./spectrograms")
TRAIN_DIR = SPECTROGRAM_DIR / "train"
VAL_DIR = SPECTROGRAM_DIR / "val"
TEST_DIR = SPECTROGRAM_DIR / "test"
NUM_WORKERS = max(1, cpu_count() - 2)


In [4]:
# Load dataset
print("Loading dataset...")
dataset = load_dataset("Hibou-Foundation/big_ds_4_raw_wav_balanced")
print(f"\nInitial dataset structure: {dataset}")
print(f"Initial features: {dataset['train'].features}")
dataset = dataset.cast_column("label", ClassLabel(names=LABELS))

sample_item = dataset['train'][0]
print(f"\nSample item keys: {sample_item.keys()}")
print(f"Audio type: {type(sample_item['audio'])}")

# Take only n percent if the dataset
dataset = DatasetDict({
    split: ds.select(range(int(0.001 * len(ds))))
    for split, ds in dataset.items()
})
print({k: v.shape for k, v in dataset.items()})


Loading dataset...

Initial dataset structure: DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 352116
    })
    val: Dataset({
        features: ['audio', 'label'],
        num_rows: 43580
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 43478
    })
})
Initial features: {'audio': List(Value('float32')), 'label': ClassLabel(names=['other', 'drone'])}

Sample item keys: dict_keys(['audio', 'label'])
Audio type: <class 'list'>
{'train': (352, 2), 'val': (43, 2), 'test': (43, 2)}


In [5]:
def convert_to_linear_spectrogram(batch):
    all_linear_db = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        data = np.array(audio)
        # Compute STFT
        stft = librosa.stft(data, n_fft=2048, hop_length=256)

        # Compute magnitude
        magnitude = np.abs(stft)

        # Convert to dB
        linear_db = librosa.amplitude_to_db(magnitude, ref=np.max)
        all_linear_db.append(torch.tensor(linear_db))
        all_labels.append(torch.tensor(label))

    return {
        # Convert to torch tensors
        "audio": all_linear_db,
        "label": all_labels,
    }



spectrogram_dataset = DatasetDict()
for split in dataset.keys():
    print(f"Converting to spectrogram split: {split}")
    spectrogram_split = dataset[split].map(
        convert_to_linear_spectrogram,
        batched=True,
        num_proc=NUM_WORKERS,
        batch_size=32,
        remove_columns=dataset[split].column_names,
    )
    spectrogram_dataset[split] = spectrogram_split

Converting to spectrogram split: train


Map (num_proc=22): 100%|██████████| 352/352 [00:04<00:00, 77.97 examples/s] 


Converting to spectrogram split: val


Map (num_proc=22): 100%|██████████| 43/43 [00:02<00:00, 18.47 examples/s]


Converting to spectrogram split: test


Map (num_proc=22): 100%|██████████| 43/43 [00:02<00:00, 18.12 examples/s]


In [6]:
# Create directory structure for YOLO classification format
def create_yolo_structure():
    for split in ['train', 'val', 'test']:
        for _label in LABELS:
            (SPECTROGRAM_DIR / split / _label).mkdir(parents=True, exist_ok=True)

create_yolo_structure()


In [7]:
def save_spectogram_to_file(spectrogram, path, width=640, height=640, cmap='viridis'):
    spec_array = np.array(spectrogram)

    # Create figure with specified dimensions
    dpi = 100
    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)

    # Create axes that fill the entire figure
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)

    # Display spectrogram with colormap
    librosa.display.specshow(
        spec_array,
        sr=SAMPLING_RATE,
        hop_length=256,
        cmap=cmap,
        ax=ax
    )
    fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close(fig)

    return spec_array

# Test the function
if 'spectrogram_dataset' in globals() and len(spectrogram_dataset['train']) > 0:
    test_path = "./test.png"
    save_spectogram_to_file(spectrogram_dataset['train'][0]['audio'], test_path)
    print(f"Test image saved to: {test_path}")
else:
    print("spectrogram_dataset not available for testing")

Test image saved to: ./test.png


In [8]:
print("Saving spectrogram images to disk...")
for split in ['train', 'val', 'test']:
    split_data = spectrogram_dataset[split]
    for idx in tqdm(range(len(split_data)), desc=f"Processing {split} set"):
        item = split_data[idx]
        spectrogram = item['audio']
        label = LABELS[item['label']]

        # Define file path
        file_path = SPECTROGRAM_DIR / split / label / f"{idx:05d}_{split}.png"
        save_spectogram_to_file(spectrogram, file_path)

Saving spectrogram images to disk...


Processing test set: 100%|██████████| 43/43 [00:02<00:00, 17.45it/s]


In [9]:
# Verify spectrogram generation
print("\nVerifying spectrogram generation...")
for split in ['train', 'val', 'test']:
    split_path = SPECTROGRAM_DIR / split
    if split_path.exists():
        for label in LABELS:
            label_path = split_path / label
            if label_path.exists():
                num_files = len(list(label_path.glob("*.png")))
                print(f"{split}/{label}: {num_files} spectrograms")



Verifying spectrogram generation...
train/other: 178 spectrograms
train/drone: 174 spectrograms
val/other: 26 spectrograms
val/drone: 17 spectrograms
test/other: 21 spectrograms
test/drone: 22 spectrograms


In [10]:
selected_size = "nano"
selected_version = "11"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}

model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model_path = Path('./models/') / model_name
model = YOLO(model_path, task="detect")

In [15]:
training_args = {
    "data": "./spectrograms",
    "epochs": 5,
    "imgsz": 640,
}

In [16]:
# Train the model
print("Starting training...")
print("=" * 60)

results = model.train(**training_args)

print("\n" + "=" * 60)
print("Training completed!")
print(f"Results saved to: {results.save_dir}")


Starting training...
Ultralytics 8.4.7 🚀 Python-3.13.9 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 7783MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./spectrograms, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

RuntimeError: Dataset 'spectrograms' error ❌ [Errno 21] Is a directory: './spectrograms'

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")

# For classification, use the test directory directly
test_results = model.val(data=str(SPECTROGRAM_DIR / 'test'))

print("\nTest Results:")
if hasattr(test_results, 'top1'):
    print(f"Top-1 Accuracy: {test_results.top1:.4f}")
if hasattr(test_results, 'top5'):
    print(f"Top-5 Accuracy: {test_results.top5:.4f}")
if hasattr(test_results, 'metrics'):
    print(f"Metrics: {test_results.metrics}")

# Alternative: Manual evaluation
print("\nManual evaluation on test set...")
test_other = list((SPECTROGRAM_DIR / 'test' / 'other').glob("*.png"))
test_drone = list((SPECTROGRAM_DIR / 'test' / 'drone').glob("*.png"))
print(f"Test samples - Other: {len(test_other)}, Drone: {len(test_drone)}")


In [ ]:
# Save the trained model
model_path = './drone_audio_yolo26.pt'
model.export(format='onnx')  # Export to ONNX for deployment
model.save(model_path)
print(f"Model saved to: {model_path}")


In [ ]:
# Example inference function
def predict_audio(audio_path, model_path='./drone_audio_yolo26.pt'):
    """
    Predict drone/other classification for an audio file.

    Args:
        audio_path: Path to audio file
        model_path: Path to trained YOLO model

    Returns:
        Prediction results
    """
    # Load model
    model = YOLO(model_path)

    # Load audio
    audio, sr = librosa.load(audio_path, sr=SAMPLING_RATE)

    # Convert to spectrogram
    temp_spec_path = './temp_spectrogram.png'
    audio_to_spectrogram_image(audio, temp_spec_path)

    # Predict
    results = model(temp_spec_path)

    # Clean up
    if os.path.exists(temp_spec_path):
        os.remove(temp_spec_path)

    return results

print("Inference function defined!")
print("\nTo use: results = predict_audio('path/to/audio.wav')")


In [ ]:
# Visualize sample spectrograms
import matplotlib.pyplot as plt

def visualize_sample_spectrograms(num_samples=4):
    """Visualize sample spectrograms from each class."""
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))

    for label_idx, label in enumerate(LABELS):
        label_path = SPECTROGRAM_DIR / 'train' / label
        if label_path.exists():
            spec_files = list(label_path.glob("*.png"))[:num_samples]

            for idx, spec_file in enumerate(spec_files):
                img = Image.open(spec_file)
                axes[label_idx, idx].imshow(img, cmap='viridis', aspect='auto')
                axes[label_idx, idx].set_title(f"{label} - {spec_file.name}")
                axes[label_idx, idx].axis('off')

    plt.suptitle('Sample Spectrograms', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Uncomment to visualize
# visualize_sample_spectrograms()


## Pipeline Summary

This notebook implements a complete pipeline for **drone audio acoustic classification** using YOLO26:

### Steps:
1. **Dataset Loading**: Loads audio dataset from HuggingFace (`Hibou-Foundation/big_ds_4_raw_wav_balanced`)
2. **Audio Preprocessing**: Converts audio to mel-spectrograms (images) using librosa
3. **Data Organization**: Organizes spectrograms in YOLO classification format (`train/class`, `val/class`, `test/class`)
4. **Model Training**: Trains YOLO26 (or YOLOv8) classification model on spectrograms
5. **Evaluation**: Evaluates model performance on test set
6. **Inference**: Provides function for predicting on new audio files

### Key Features:
- Binary classification: `other` vs `drone`
- Mel-spectrogram conversion with configurable parameters
- YOLO26/YOLOv8 classification model
- Complete training pipeline with validation
- Model export for deployment (ONNX format)

### Outputs:
- Trained model: `drone_audio_yolo26.pt`
- Spectrograms: `./spectrograms/`
- Training logs: `./runs/classify/drone_audio_classification/`
- Dataset config: `./spectrograms/dataset.yaml`


In [34]:
from datasets import load_dataset
import ultralytics
import os
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset, DatasetDict, ClassLabel, Audio as DatasetAudio
from PIL import Image
import shutil
from tqdm.auto import tqdm
from ultralytics import YOLO
import yaml
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import matplotlib.pyplot as plt
import librosa.display



In [ ]:
ultralytics.checks()

In [10]:
SAMPLING_RATE = 16000
LABELS = ['other', 'drone']

# Paths
SPECTROGRAM_DIR = Path("./spectrograms")
TRAIN_DIR = SPECTROGRAM_DIR / "train"
VAL_DIR = SPECTROGRAM_DIR / "val"
TEST_DIR = SPECTROGRAM_DIR / "test"
NUM_WORKERS = max(1, cpu_count() - 2)


In [19]:
# Load dataset
print("Loading dataset...")
dataset = load_dataset("Hibou-Foundation/big_ds_4_raw_wav_balanced")
print(f"\nInitial dataset structure: {dataset}")
print(f"Initial features: {dataset['train'].features}")
dataset = dataset.cast_column("label", ClassLabel(names=LABELS))

sample_item = dataset['train'][0]
print(f"\nSample item keys: {sample_item.keys()}")
print(f"Audio type: {type(sample_item['audio'])}")

# Take only n percent if the dataset
dataset = DatasetDict({
    split: ds.select(range(int(0.01 * len(ds))))
    for split, ds in dataset.items()
})
print({k: v.shape for k, v in dataset.items()})


Loading dataset...

Initial dataset structure: DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 352116
    })
    val: Dataset({
        features: ['audio', 'label'],
        num_rows: 43580
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 43478
    })
})
Initial features: {'audio': List(Value('float32')), 'label': ClassLabel(names=['other', 'drone'])}

Sample item keys: dict_keys(['audio', 'label'])
Audio type: <class 'list'>
{'train': (3521, 2), 'val': (435, 2), 'test': (434, 2)}


In [20]:
def convert_to_linear_spectrogram(batch):
    all_linear_db = []
    all_labels = []

    for audio, label in zip(batch["audio"], batch["label"]):
        data = np.array(audio)
        # Compute STFT
        stft = librosa.stft(data, n_fft=2048, hop_length=256)

        # Compute magnitude
        magnitude = np.abs(stft)

        # Convert to dB
        linear_db = librosa.amplitude_to_db(magnitude, ref=np.max)
        all_linear_db.append(torch.tensor(linear_db))
        all_labels.append(torch.tensor(label))

    return {
        # Convert to torch tensors
        "audio": all_linear_db,
        "label": all_labels,
    }



spectrogram_dataset = DatasetDict()
for split in dataset.keys():
    print(f"Converting to spectrogram split: {split}")
    spectrogram_split = dataset[split].map(
        convert_to_linear_spectrogram,
        batched=True,
        num_proc=NUM_WORKERS,
        batch_size=32,
        remove_columns=dataset[split].column_names,
    )
    spectrogram_dataset[split] = spectrogram_split

Converting to spectrogram split: train


Map (num_proc=22): 100%|██████████| 3521/3521 [00:18<00:00, 189.19 examples/s]


Converting to spectrogram split: val


Map (num_proc=22): 100%|██████████| 435/435 [00:05<00:00, 77.43 examples/s] 


Converting to spectrogram split: test


Map (num_proc=22): 100%|██████████| 434/434 [00:05<00:00, 84.69 examples/s] 


In [22]:
# Create directory structure for YOLO classification format
def create_yolo_structure():
    for split in ['train', 'val', 'test']:
        for _label in LABELS:
            (SPECTROGRAM_DIR / split / _label).mkdir(parents=True, exist_ok=True)

create_yolo_structure()


In [38]:
def save_spectogram_to_file(spectrogram, path, width=640, height=640, cmap='viridis'):
    spec_array = np.array(spectrogram)
    
    # Create figure with specified dimensions
    dpi = 100
    fig = plt.figure(figsize=(width / dpi, height / dpi), dpi=dpi)
    
    # Create axes that fill the entire figure
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    
    # Display spectrogram with colormap
    librosa.display.specshow(
        spec_array,
        sr=SAMPLING_RATE,
        hop_length=256,
        cmap=cmap,
        ax=ax
    )
    fig.savefig(path, dpi=dpi, bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    
    return spec_array

# Test the function
if 'spectrogram_dataset' in globals() and len(spectrogram_dataset['train']) > 0:
    test_path = "./test.png"
    save_spectogram_to_file(spectrogram_dataset['train'][0]['audio'], test_path)
    print(f"Test image saved to: {test_path}")
else:
    print("spectrogram_dataset not available for testing")

Test image saved to: ./test.png


In [39]:
print("Saving spectrogram images to disk...")
for split in ['train', 'val', 'test']:
    split_data = spectrogram_dataset[split]
    for idx in tqdm(range(len(split_data)), desc=f"Processing {split} set"):
        item = split_data[idx]
        spectrogram = item['audio']
        label = LABELS[item['label']]

        # Define file path
        file_path = SPECTROGRAM_DIR / split / label / f"{split}_{idx:05d}.png"

        # Save spectrogram image
        save_spectogram_to_file(spectrogram, file_path)

Saving spectrogram images to disk...


Processing test set: 100%|██████████| 434/434 [00:28<00:00, 15.20it/s]


In [40]:
# Verify spectrogram generation
print("\nVerifying spectrogram generation...")
for split in ['train', 'val', 'test']:
    split_path = SPECTROGRAM_DIR / split
    if split_path.exists():
        for label in LABELS:
            label_path = split_path / label
            if label_path.exists():
                num_files = len(list(label_path.glob("*.png")))
                print(f"{split}/{label}: {num_files} spectrograms")



Verifying spectrogram generation...
train/other: 1735 spectrograms
train/drone: 1786 spectrograms
val/other: 223 spectrograms
val/drone: 212 spectrograms
test/other: 224 spectrograms
test/drone: 210 spectrograms


In [41]:
# Create YOLO dataset configuration file
def create_yolo_config():
    """Create YOLO dataset configuration file."""
    config = {
        'path': str(SPECTROGRAM_DIR.absolute()),
        'train': 'train',
        'val': 'val',
        'test': 'test',
        'names': {0: 'other', 1: 'drone'},
        'nc': 2
    }
    
    config_path = SPECTROGRAM_DIR / 'dataset.yaml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✓ Dataset config saved to {config_path}")
    return config_path

config_path = create_yolo_config()


✓ Dataset config saved to spectrograms/dataset.yaml


In [56]:
selected_size = "nano"
selected_version = "11"

YOLO_MODEL_SIZE = {
    "nano": "n",
    "small": "s",
    "medium": "m",
    "large": "l",
    "xlarge": "x",
}

model_name = f"yolo{selected_version}{YOLO_MODEL_SIZE[selected_size]}.pt"
model_path = Path('./models/') / model_name
model = YOLO(model_path, task="detect")

In [45]:
# Training configuration for classification
# Note: YOLO classification uses simpler parameters than detection
training_args = {
    'data': "./spectograms/",
    'epochs': 5,
    'imgsz': 224,  # Image size for classification
    'batch': 32,
    'lr0': 0.01,  # Initial learning rate
    'lrf': 0.1,  # Final learning rate (lr0 * lrf)
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    'project': './runs/classify',
    'name': 'drone_audio_classification',
    'exist_ok': True,
    'pretrained': True,
    'optimizer': 'SGD',  # or 'Adam', 'AdamW'
    'verbose': True,
    'seed': 42,
    'deterministic': True,
    'cos_lr': False,  # Cosine learning rate scheduler
    'resume': False,
    'amp': True,  # Automatic Mixed Precision
    'fraction': 1.0,  # Dataset fraction to use
    'profile': False,
    'freeze': None,  # Freeze layers: [0, 1, 2] or None
    'val': True,  # Validate during training
}

print("Training configuration set!")
print(f"Training on: {training_args['data']}")
print(f"Epochs: {training_args['epochs']}")
print(f"Batch size: {training_args['batch']}")
print(f"Image size: {training_args['imgsz']}")
print(f"Learning rate: {training_args['lr0']} -> {training_args['lr0'] * training_args['lrf']}")


Training configuration set!
Training on: ./spectograms/
Epochs: 5
Batch size: 32
Image size: 224
Learning rate: 0.01 -> 0.001


In [46]:
# Train the model
print("Starting training...")
print("=" * 60)

results = model.train(**training_args)

print("\n" + "=" * 60)
print("Training completed!")
print(f"Results saved to: {results.save_dir}")


Starting training...
New https://pypi.org/project/ultralytics/8.4.7 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.13.9 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 7783MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./spectograms/, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=drone_

######################################################################## 100.0%


RuntimeError: Dataset 'spectograms' error ❌ [Errno 2] No such file or directory: '/home/pierre/Documents/Projects/PST4/datasets/spectograms/train'

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")

# For classification, use the test directory directly
test_results = model.val(data=str(SPECTROGRAM_DIR / 'test'))

print("\nTest Results:")
if hasattr(test_results, 'top1'):
    print(f"Top-1 Accuracy: {test_results.top1:.4f}")
if hasattr(test_results, 'top5'):
    print(f"Top-5 Accuracy: {test_results.top5:.4f}")
if hasattr(test_results, 'metrics'):
    print(f"Metrics: {test_results.metrics}")

# Alternative: Manual evaluation
print("\nManual evaluation on test set...")
test_other = list((SPECTROGRAM_DIR / 'test' / 'other').glob("*.png"))
test_drone = list((SPECTROGRAM_DIR / 'test' / 'drone').glob("*.png"))
print(f"Test samples - Other: {len(test_other)}, Drone: {len(test_drone)}")


In [ ]:
# Save the trained model
model_path = './drone_audio_yolo26.pt'
model.export(format='onnx')  # Export to ONNX for deployment
model.save(model_path)
print(f"Model saved to: {model_path}")


In [ ]:
# Example inference function
def predict_audio(audio_path, model_path='./drone_audio_yolo26.pt'):
    """
    Predict drone/other classification for an audio file.
    
    Args:
        audio_path: Path to audio file
        model_path: Path to trained YOLO model
    
    Returns:
        Prediction results
    """
    # Load model
    model = YOLO(model_path)
    
    # Load audio
    audio, sr = librosa.load(audio_path, sr=SAMPLING_RATE)
    
    # Convert to spectrogram
    temp_spec_path = './temp_spectrogram.png'
    audio_to_spectrogram_image(audio, temp_spec_path)
    
    # Predict
    results = model(temp_spec_path)
    
    # Clean up
    if os.path.exists(temp_spec_path):
        os.remove(temp_spec_path)
    
    return results

print("Inference function defined!")
print("\nTo use: results = predict_audio('path/to/audio.wav')")


In [ ]:
# Visualize sample spectrograms
import matplotlib.pyplot as plt

def visualize_sample_spectrograms(num_samples=4):
    """Visualize sample spectrograms from each class."""
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
    
    for label_idx, label in enumerate(LABELS):
        label_path = SPECTROGRAM_DIR / 'train' / label
        if label_path.exists():
            spec_files = list(label_path.glob("*.png"))[:num_samples]
            
            for idx, spec_file in enumerate(spec_files):
                img = Image.open(spec_file)
                axes[label_idx, idx].imshow(img, cmap='viridis', aspect='auto')
                axes[label_idx, idx].set_title(f"{label} - {spec_file.name}")
                axes[label_idx, idx].axis('off')
    
    plt.suptitle('Sample Spectrograms', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Uncomment to visualize
# visualize_sample_spectrograms()


## Pipeline Summary

This notebook implements a complete pipeline for **drone audio acoustic classification** using YOLO26:

### Steps:
1. **Dataset Loading**: Loads audio dataset from HuggingFace (`Hibou-Foundation/big_ds_4_raw_wav_balanced`)
2. **Audio Preprocessing**: Converts audio to mel-spectrograms (images) using librosa
3. **Data Organization**: Organizes spectrograms in YOLO classification format (`train/class`, `val/class`, `test/class`)
4. **Model Training**: Trains YOLO26 (or YOLOv8) classification model on spectrograms
5. **Evaluation**: Evaluates model performance on test set
6. **Inference**: Provides function for predicting on new audio files

### Key Features:
- Binary classification: `other` vs `drone`
- Mel-spectrogram conversion with configurable parameters
- YOLO26/YOLOv8 classification model
- Complete training pipeline with validation
- Model export for deployment (ONNX format)

### Outputs:
- Trained model: `drone_audio_yolo26.pt`
- Spectrograms: `./spectrograms/`
- Training logs: `./runs/classify/drone_audio_classification/`
- Dataset config: `./spectrograms/dataset.yaml`
